# Phone Black Box — Master Dataset Creation Notebook

## Purpose

This notebook is a **faithful, explained reconstruction of the exact Python script that created the `master_dataset`**.

The objective is not to invent a new dataset pipeline. It documents what the supplied builder script actually does:

```text
VZCrash
   +
Isolated Data / Combined Anomalies
   +
Isolated Normal Road
   +
road_accident_imu_dataset_8000.csv
        ↓
unit conversion
        ↓
100-sample windows
        ↓
feature extraction
        ↓
label assignment
        ↓
train / val / test assignment
        ↓
master_driving_events.parquet / CSV
        ↓
train / val / test files
        ↓
label_mapping.json
feature_metadata.json
```

### Very important finding

The script header says **51 features**, but the actual `extract_features_from_window()` function creates **62 features**.

That is not a guess. Counting the implemented feature groups gives:

- 30 per-axis statistical features
- 9 acceleration magnitude / horizontal features
- 3 jerk features
- 4 gyro magnitude features
- 2 speed features
- 3 FFT energy features
- 1 spectral entropy feature
- 4 skew/kurtosis features
- 6 correlation features

**Total = 62 features.**

This explains why the processed `master_dataset` you supplied contains 62 model features.


## 1. Imports

The original script imports:

- `os` and `glob` → filesystem and file discovery
- `json` → metadata files
- `time` → execution-time measurement
- `numpy` → numerical processing and sensor calculations
- `pandas` → CSV/Parquet processing and tabular output
- `scipy.signal` → imported by the original script
- `scipy.stats.skew` and `kurtosis` → advanced statistical features

The code below is the same import layer used by the builder.


In [1]:
import os
import glob
import json
import time

import numpy as np
import pandas as pd

from scipy import signal
from scipy.stats import skew, kurtosis


## 2. Reproducibility and project paths

The builder uses a deterministic NumPy random generator:

```python
RNG = np.random.RandomState(42)
```

This matters because the script randomly samples VZCrash records, randomly chooses normal-driving windows, shuffles anomaly/supplemental records, and adds synthetic noise to the 8,000-record IMU dataset.

The expected directory structure is:

```text
project/
├── dataset/
│   ├── VZCrash/
│   │   ├── train_data/
│   │   ├── validation_data/
│   │   └── test_data/
│   ├── _extracted/
│   │   ├── combined_anomalies/
│   │   └── combined_normal/
│   ├── road_accident_imu_dataset_8000.csv
│   └── master_dataset/
└── <this notebook/script>
```


In [2]:
# ============================================================
# 2. Reproducibility and project paths — Jupyter/VS Code safe
# ============================================================

from pathlib import Path

RNG = np.random.RandomState(42)

def find_dataset_dir():
    """Find the project's dataset directory from the notebook working directory."""

    candidates = []

    # Most common VS Code/Jupyter case: notebook opened from project root.
    cwd = Path.cwd().resolve()
    candidates.append(cwd / "dataset")

    # Also check all parents.
    for parent in cwd.parents:
        candidates.append(parent / "dataset")

    # If a real __file__ exists, also inspect its parents.
    if "__file__" in globals():
        try:
            file_path = Path(__file__).resolve()
            candidates.append(file_path.parent / "dataset")
            candidates.extend(parent / "dataset" for parent in file_path.parent.parents)
        except Exception:
            pass

    for candidate in candidates:
        if candidate.is_dir():
            return candidate

    checked = "\n".join(str(x) for x in candidates)
    raise FileNotFoundError(
        "Could not find the project's 'dataset' directory.\n"
        "The notebook is probably running with the wrong working directory.\n\n"
        "Checked:\n" + checked +
        "\n\nIn VS Code, open the 'Phone Black Box copy' project folder "
        "as the workspace and run the notebook from there."
    )

DATASET_DIR = find_dataset_dir()
PROJECT_ROOT = DATASET_DIR.parent
OUTPUT_DIR = DATASET_DIR / "master_dataset"
ANOMALIES_DIR = DATASET_DIR / "_extracted" / "combined_anomalies"
NORMAL_ROAD_DIR = DATASET_DIR / "_extracted" / "combined_normal"

print("=" * 70)
print("DATASET PATH CONFIGURATION")
print("=" * 70)
print("Current working directory:", Path.cwd().resolve())
print("Project root             :", PROJECT_ROOT)
print("Dataset directory        :", DATASET_DIR)
print("Output directory         :", OUTPUT_DIR)
print("Anomalies directory      :", ANOMALIES_DIR)
print("Normal-road directory    :", NORMAL_ROAD_DIR)

print("\nSource availability:")
print("VZCrash                   :", (DATASET_DIR / "VZCrash").exists())
print("Isolated Data             :", (DATASET_DIR / "Isolated Data").exists())
print("Combined anomalies       :", ANOMALIES_DIR.exists())
print("Combined normal          :", NORMAL_ROAD_DIR.exists())
print("8000-row IMU CSV         :", (DATASET_DIR / "road_accident_imu_dataset_8000.csv").exists())


DATASET PATH CONFIGURATION
Current working directory: /Users/viditagarwal/Desktop/Phone Black Box
Project root             : /Users/viditagarwal/Desktop/Phone Black Box
Dataset directory        : /Users/viditagarwal/Desktop/Phone Black Box/dataset
Output directory         : /Users/viditagarwal/Desktop/Phone Black Box/dataset/master_dataset
Anomalies directory      : /Users/viditagarwal/Desktop/Phone Black Box/dataset/_extracted/combined_anomalies
Normal-road directory    : /Users/viditagarwal/Desktop/Phone Black Box/dataset/_extracted/combined_normal

Source availability:
VZCrash                   : True
Isolated Data             : True
Combined anomalies       : True
Combined normal          : True
8000-row IMU CSV         : True


## 3. Label ontology

The builder converts all usable event types into one common nine-class ontology.

| ID | Class |
|---:|---|
| 0 | crash |
| 1 | near_miss |
| 2 | normal_driving |
| 3 | braking |
| 4 | hard_braking |
| 5 | turn |
| 6 | sharp_turn |
| 7 | bump |
| 8 | pothole |

This is the classification target used by the final master dataset.


In [3]:
LABEL_MAPPING = {
    0: "crash",
    1: "near_miss",
    2: "normal_driving",
    3: "braking",
    4: "hard_braking",
    5: "turn",
    6: "sharp_turn",
    7: "bump",
    8: "pothole",
}

INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}

WINDOW_SIZE = 100       # 100 samples
DT = 0.01               # 100 Hz -> 0.01 seconds/sample

print("Classes:", LABEL_MAPPING)
print("Window:", WINDOW_SIZE, "samples =", WINDOW_SIZE * DT, "seconds")


Classes: {0: 'crash', 1: 'near_miss', 2: 'normal_driving', 3: 'braking', 4: 'hard_braking', 5: 'turn', 6: 'sharp_turn', 7: 'bump', 8: 'pothole'}
Window: 100 samples = 1.0 seconds


# 4. Feature Engineering — the most important function

`extract_features_from_window()` is the heart of the master dataset builder.

### Input

Each window contains:

```text
acc  -> 100 × 3
       x, y, z acceleration in m/s²

gyro -> 100 × 3
       x, y, z angular velocity in deg/s

speed_mean  -> km/h
speed_delta -> km/h/s
```

### Output

One flat dictionary representing that complete 1-second window.

The feature extractor creates **62 features**, despite the old docstring saying 51.


In [4]:
def extract_features_from_window(acc, gyro, speed_mean=0.0, speed_delta=0.0):
    """
    Extract features from one 100-sample sensor window.

    acc:
        shape (100, 3), m/s²

    gyro:
        shape (100, 3), deg/s

    speed_mean:
        km/h

    speed_delta:
        km/h/s

    Returns:
        Dictionary containing 62 features.
    """

    feat = {}

    # --------------------------------------------------------
    # A. Per-axis statistical features
    # --------------------------------------------------------
    # 6 sensor axes × 5 statistics = 30 features
    for idx, axis in enumerate(["x", "y", "z"]):
        a = acc[:, idx]
        g = gyro[:, idx]

        feat[f"acc_{axis}_mean"] = float(np.mean(a))
        feat[f"acc_{axis}_std"] = float(np.std(a))
        feat[f"acc_{axis}_min"] = float(np.min(a))
        feat[f"acc_{axis}_max"] = float(np.max(a))
        feat[f"acc_{axis}_p2p"] = float(np.ptp(a))

        feat[f"gyro_{axis}_mean"] = float(np.mean(g))
        feat[f"gyro_{axis}_std"] = float(np.std(g))
        feat[f"gyro_{axis}_min"] = float(np.min(g))
        feat[f"gyro_{axis}_max"] = float(np.max(g))
        feat[f"gyro_{axis}_p2p"] = float(np.ptp(g))

    # --------------------------------------------------------
    # B. Magnitude features
    # --------------------------------------------------------
    acc_mag = np.linalg.norm(acc, axis=1)
    gyro_mag = np.linalg.norm(gyro, axis=1)

    # Horizontal x-y acceleration magnitude
    acc_horiz = np.linalg.norm(acc[:, :2], axis=1)

    feat["acc_mag_mean"] = float(np.mean(acc_mag))
    feat["acc_mag_std"] = float(np.std(acc_mag))
    feat["acc_mag_min"] = float(np.min(acc_mag))
    feat["acc_mag_max"] = float(np.max(acc_mag))
    feat["acc_mag_p2p"] = float(np.ptp(acc_mag))
    feat["acc_mag_rms"] = float(np.sqrt(np.mean(acc_mag**2)))

    feat["acc_horiz_mean"] = float(np.mean(acc_horiz))
    feat["acc_horiz_std"] = float(np.std(acc_horiz))
    feat["acc_horiz_max"] = float(np.max(acc_horiz))

    # --------------------------------------------------------
    # C. Jerk
    # --------------------------------------------------------
    # Jerk = change in acceleration / time.
    jerk = np.diff(acc_mag) / DT

    feat["jerk_mean"] = (
        float(np.mean(np.abs(jerk))) if len(jerk) > 0 else 0.0
    )
    feat["jerk_std"] = (
        float(np.std(jerk)) if len(jerk) > 0 else 0.0
    )
    feat["jerk_max"] = (
        float(np.max(np.abs(jerk))) if len(jerk) > 0 else 0.0
    )

    # --------------------------------------------------------
    # D. Angular velocity magnitude
    # --------------------------------------------------------
    feat["gyro_mag_mean"] = float(np.mean(gyro_mag))
    feat["gyro_mag_std"] = float(np.std(gyro_mag))
    feat["gyro_mag_max"] = float(np.max(gyro_mag))
    feat["gyro_mag_p2p"] = float(np.ptp(gyro_mag))

    # --------------------------------------------------------
    # E. Speed
    # --------------------------------------------------------
    feat["speed_mean"] = float(speed_mean)
    feat["speed_delta"] = float(speed_delta)

    # --------------------------------------------------------
    # F. Frequency-domain features
    # --------------------------------------------------------
    centered_acc = acc_mag - np.mean(acc_mag)

    fft_vals = np.abs(np.fft.rfft(centered_acc))

    feat["fft_energy_total"] = float(np.sum(fft_vals**2))

    n_freq = len(fft_vals)

    # Original implementation:
    # first quarter of FFT bins = low band
    feat["fft_energy_low"] = float(
        np.sum(fft_vals[:max(1, n_freq // 4)]**2)
    )

    feat["fft_energy_high"] = float(
        np.sum(fft_vals[max(1, n_freq // 4):]**2)
    )

    # --------------------------------------------------------
    # G. Spectral entropy
    # --------------------------------------------------------
    psd = (fft_vals**2) / n_freq
    psd_norm = psd / (np.sum(psd) + 1e-9)

    feat["fft_entropy"] = float(
        -np.sum(psd_norm * np.log2(psd_norm + 1e-9))
    )

    # --------------------------------------------------------
    # H. Advanced statistical features
    # --------------------------------------------------------
    feat["acc_mag_skew"] = (
        float(skew(acc_mag.astype(np.float64), bias=False))
        if np.std(acc_mag) > 1e-4 else 0.0
    )

    feat["acc_mag_kurtosis"] = (
        float(kurtosis(acc_mag.astype(np.float64), bias=False))
        if np.std(acc_mag) > 1e-4 else 0.0
    )

    feat["gyro_mag_skew"] = (
        float(skew(gyro_mag.astype(np.float64), bias=False))
        if np.std(gyro_mag) > 1e-4 else 0.0
    )

    feat["gyro_mag_kurtosis"] = (
        float(kurtosis(gyro_mag.astype(np.float64), bias=False))
        if np.std(gyro_mag) > 1e-4 else 0.0
    )

    # --------------------------------------------------------
    # I. Axis correlations
    # --------------------------------------------------------
    feat["acc_xy_corr"] = (
        float(np.corrcoef(acc[:, 0], acc[:, 1])[0, 1])
        if np.std(acc[:, 0]) > 1e-4 and np.std(acc[:, 1]) > 1e-4
        else 0.0
    )

    feat["acc_xz_corr"] = (
        float(np.corrcoef(acc[:, 0], acc[:, 2])[0, 1])
        if np.std(acc[:, 0]) > 1e-4 and np.std(acc[:, 2]) > 1e-4
        else 0.0
    )

    feat["acc_yz_corr"] = (
        float(np.corrcoef(acc[:, 1], acc[:, 2])[0, 1])
        if np.std(acc[:, 1]) > 1e-4 and np.std(acc[:, 2]) > 1e-4
        else 0.0
    )

    feat["gyro_xy_corr"] = (
        float(np.corrcoef(gyro[:, 0], gyro[:, 1])[0, 1])
        if np.std(gyro[:, 0]) > 1e-4 and np.std(gyro[:, 1]) > 1e-4
        else 0.0
    )

    feat["gyro_xz_corr"] = (
        float(np.corrcoef(gyro[:, 0], gyro[:, 2])[0, 1])
        if np.std(gyro[:, 0]) > 1e-4 and np.std(gyro[:, 2]) > 1e-4
        else 0.0
    )

    feat["gyro_yz_corr"] = (
        float(np.corrcoef(gyro[:, 1], gyro[:, 2])[0, 1])
        if np.std(gyro[:, 1]) > 1e-4 and np.std(gyro[:, 2]) > 1e-4
        else 0.0
    )

    # Exact feature-count check.
    assert len(feat) == 62, f"Expected 62 features, got {len(feat)}"

    return feat


## 5. Verify the feature extractor independently

Before processing thousands of files, the original script's feature function should be tested on synthetic sensor data.

This catches accidental changes to feature count or naming before the expensive dataset-building stage.


In [5]:
# Synthetic smoke test
rng = np.random.RandomState(42)

test_acc = rng.normal(size=(100, 3))
test_gyro = rng.normal(size=(100, 3))

test_features = extract_features_from_window(
    test_acc,
    test_gyro,
    speed_mean=40.0,
    speed_delta=-3.0
)

print("Generated features:", len(test_features))
print("First 10:", list(test_features.keys())[:10])
print("✓ Feature extractor smoke test passed.")


Generated features: 62
First 10: ['acc_x_mean', 'acc_x_std', 'acc_x_min', 'acc_x_max', 'acc_x_p2p', 'gyro_x_mean', 'gyro_x_std', 'gyro_x_min', 'gyro_x_max', 'gyro_x_p2p']
✓ Feature extractor smoke test passed.


# 6. VZCrash processing

The VZCrash function processes Parquet files and creates samples for:

- crash → class 0
- near_miss → class 1
- normal_driving → class 2
- braking → class 3
- hard_braking → class 4
- turn → class 5
- sharp_turn → class 6

The original implementation limits each class to `max_per_class`.

### Important source transformations

For each VZCrash event:

```text
gsensor × 9.80665 → m/s²
gyro                  → deg/s
gps_speed             → speed array
```

The script then derives event windows differently depending on the class.


In [6]:
def process_vzcrash_split(split_name, parquet_files, max_per_class=1750):
    """Process VZCrash exactly according to the supplied builder logic."""

    records = []
    class_counts = {c: 0 for c in [0, 1, 2, 3, 4, 5, 6]}

    print(
        f"\nProcessing VZCrash {split_name} "
        f"({len(parquet_files)} files, target ~{max_per_class}/class)..."
    )

    for fidx, fpath in enumerate(parquet_files):

        if all(class_counts[c] >= max_per_class for c in class_counts):
            break

        print(
            f"  Reading file {fidx + 1}/{len(parquet_files)}: "
            f"{os.path.basename(fpath)}"
        )

        df = pd.read_parquet(fpath)

        # Deterministic shuffle
        indices = RNG.permutation(len(df))

        for idx in indices:

            row = df.iloc[idx]
            lbl = row["label"]

            acc_raw = np.stack(row["gsensor"]) * 9.80665
            gyro_raw = np.stack(row["gyro"])
            speed = np.array(row["gps_speed"], dtype=float)

            ds = np.diff(speed) if len(speed) > 1 else np.array([0.0])
            min_ds = float(ds.min()) if len(ds) > 0 else 0.0
            avg_speed = float(np.mean(speed)) if len(speed) > 0 else 0.0

            # ------------------------------------------------
            # Crash
            # ------------------------------------------------
            if lbl == 0 and class_counts[0] < max_per_class:

                acc_mag = np.linalg.norm(acc_raw, axis=1)
                t_peak = int(np.argmax(acc_mag))

                start = max(
                    0,
                    min(
                        len(acc_raw) - WINDOW_SIZE,
                        t_peak - WINDOW_SIZE // 2
                    )
                )

                w_acc = acc_raw[start:start + WINDOW_SIZE]
                w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                feat = extract_features_from_window(
                    w_acc, w_gyro, avg_speed, min_ds
                )

                feat["class_label"] = 0
                feat["class_name"] = "crash"
                feat["source_dataset"] = "VZCrash"
                feat["split"] = split_name

                records.append(feat)
                class_counts[0] += 1

            # ------------------------------------------------
            # Near miss
            # ------------------------------------------------
            elif lbl == 1 and class_counts[1] < max_per_class:

                horiz_mag = np.linalg.norm(acc_raw[:, :2], axis=1)
                t_peak = int(np.argmax(horiz_mag))

                start = max(
                    0,
                    min(
                        len(acc_raw) - WINDOW_SIZE,
                        t_peak - WINDOW_SIZE // 2
                    )
                )

                w_acc = acc_raw[start:start + WINDOW_SIZE]
                w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                feat = extract_features_from_window(
                    w_acc, w_gyro, avg_speed, min_ds
                )

                feat["class_label"] = 1
                feat["class_name"] = "near_miss"
                feat["source_dataset"] = "VZCrash"
                feat["split"] = split_name

                records.append(feat)
                class_counts[1] += 1

            # ------------------------------------------------
            # Original class 2: derive maneuver subclasses
            # ------------------------------------------------
            elif lbl == 2:

                acc_mag = np.linalg.norm(acc_raw, axis=1)
                gyro_mag = np.linalg.norm(gyro_raw, axis=1)

                yaw_max = float(np.abs(gyro_raw[:, 2]).max())

                # Hard braking
                if (
                    min_ds < -14.0
                    and acc_mag.max() < 28.0
                    and class_counts[4] < max_per_class
                ):
                    t_dec = int(np.argmin(acc_raw[:, 0]))

                    start = max(
                        0,
                        min(
                            len(acc_raw) - WINDOW_SIZE,
                            t_dec - WINDOW_SIZE // 2
                        )
                    )

                    w_acc = acc_raw[start:start + WINDOW_SIZE]
                    w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                    feat = extract_features_from_window(
                        w_acc, w_gyro, avg_speed, min_ds
                    )

                    feat["class_label"] = 4
                    feat["class_name"] = "hard_braking"
                    feat["source_dataset"] = "VZCrash"
                    feat["split"] = split_name

                    records.append(feat)
                    class_counts[4] += 1

                # Braking
                elif (
                    -12.0 <= min_ds <= -4.5
                    and yaw_max < 16.0
                    and acc_mag.max() < 20.0
                    and class_counts[3] < max_per_class
                ):
                    t_dec = int(np.argmin(acc_raw[:, 0]))

                    start = max(
                        0,
                        min(
                            len(acc_raw) - WINDOW_SIZE,
                            t_dec - WINDOW_SIZE // 2
                        )
                    )

                    w_acc = acc_raw[start:start + WINDOW_SIZE]
                    w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                    feat = extract_features_from_window(
                        w_acc, w_gyro, avg_speed, min_ds
                    )

                    feat["class_label"] = 3
                    feat["class_name"] = "braking"
                    feat["source_dataset"] = "VZCrash"
                    feat["split"] = split_name

                    records.append(feat)
                    class_counts[3] += 1

                # Sharp turn
                elif (
                    yaw_max > 28.0
                    and acc_mag.max() < 25.0
                    and class_counts[6] < max_per_class
                ):
                    t_turn = int(np.argmax(np.abs(gyro_raw[:, 2])))

                    start = max(
                        0,
                        min(
                            len(acc_raw) - WINDOW_SIZE,
                            t_turn - WINDOW_SIZE // 2
                        )
                    )

                    w_acc = acc_raw[start:start + WINDOW_SIZE]
                    w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                    feat = extract_features_from_window(
                        w_acc, w_gyro, avg_speed, min_ds
                    )

                    feat["class_label"] = 6
                    feat["class_name"] = "sharp_turn"
                    feat["source_dataset"] = "VZCrash"
                    feat["split"] = split_name

                    records.append(feat)
                    class_counts[6] += 1

                # Turn
                elif (
                    12.0 <= yaw_max <= 25.0
                    and acc_mag.max() < 18.0
                    and class_counts[5] < max_per_class
                ):
                    t_turn = int(np.argmax(np.abs(gyro_raw[:, 2])))

                    start = max(
                        0,
                        min(
                            len(acc_raw) - WINDOW_SIZE,
                            t_turn - WINDOW_SIZE // 2
                        )
                    )

                    w_acc = acc_raw[start:start + WINDOW_SIZE]
                    w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                    feat = extract_features_from_window(
                        w_acc, w_gyro, avg_speed, min_ds
                    )

                    feat["class_label"] = 5
                    feat["class_name"] = "turn"
                    feat["source_dataset"] = "VZCrash"
                    feat["split"] = split_name

                    records.append(feat)
                    class_counts[5] += 1

                # Normal driving
                elif (
                    abs(min_ds) < 3.5
                    and yaw_max < 9.0
                    and abs(acc_mag.mean() - 9.81) < 2.5
                    and class_counts[2] < max_per_class
                ):
                    start = RNG.randint(
                        0,
                        len(acc_raw) - WINDOW_SIZE
                    )

                    w_acc = acc_raw[start:start + WINDOW_SIZE]
                    w_gyro = gyro_raw[start:start + WINDOW_SIZE]

                    feat = extract_features_from_window(
                        w_acc, w_gyro, avg_speed, min_ds
                    )

                    feat["class_label"] = 2
                    feat["class_name"] = "normal_driving"
                    feat["source_dataset"] = "VZCrash"
                    feat["split"] = split_name

                    records.append(feat)
                    class_counts[2] += 1

    print(f"VZCrash {split_name} summary:", class_counts)

    return records


# 7. Isolated Data — Bump and Pothole

The original script searches:

```text
dataset/_extracted/combined_anomalies/
```

It expects an `annotation_text` column and looks specifically for:

- `Bump`
- `Pothole`

For every contiguous annotation block:

1. the block must contain at least 100 samples;
2. 100-sample windows are generated;
3. windows overlap by 50 samples (`step = 50`);
4. gyroscope values are converted from rad/s to deg/s;
5. speed is not read from the recording — the original script supplies a fixed `speed_mean=35.0` and `speed_delta=0.0`;
6. each window receives class 7 or 8.


In [7]:
def process_road_anomalies(max_per_class=None):
    if max_per_class is None:
        max_per_class = {"bump": 2200, "pothole": 2200}

    print("\nProcessing Isolated Road Anomalies...")

    anomaly_csvs = sorted(
        glob.glob(os.path.join(ANOMALIES_DIR, "*/ *.csv".replace(" ", "")))
        + glob.glob(os.path.join(ANOMALIES_DIR, "*.csv"))
    )

    print("Found", len(anomaly_csvs), "anomaly CSV files.")

    bump_records = []
    pothole_records = []

    for fpath in anomaly_csvs:
        try:
            df = pd.read_csv(fpath)

            if "annotation_text" not in df.columns:
                continue

            if "totalAcceleration_x" in df.columns:
                acc_cols = [
                    "totalAcceleration_x",
                    "totalAcceleration_y",
                    "totalAcceleration_z",
                ]
            else:
                acc_cols = [
                    "accelerometer_x",
                    "accelerometer_y",
                    "accelerometer_z",
                ]

            gyro_cols = [
                "gyroscope_x",
                "gyroscope_y",
                "gyroscope_z",
            ]

            for label, records_list, class_idx in [
                ("Bump", bump_records, 7),
                ("Pothole", pothole_records, 8),
            ]:

                mask = df["annotation_text"] == label

                if not mask.any():
                    continue

                blocks = (mask != mask.shift(1)).cumsum()[mask]

                for block_id in blocks.unique():

                    if len(records_list) >= max_per_class[label.lower()]:
                        break

                    block_df = df[mask & (blocks == block_id)]

                    if len(block_df) >= WINDOW_SIZE:

                        step = WINDOW_SIZE // 2

                        for s in range(
                            0,
                            len(block_df) - WINDOW_SIZE + 1,
                            step
                        ):
                            w_acc = (
                                block_df[acc_cols]
                                .iloc[s:s + WINDOW_SIZE]
                                .values
                                .astype(float)
                            )

                            w_gyro = (
                                block_df[gyro_cols]
                                .iloc[s:s + WINDOW_SIZE]
                                .values
                                .astype(float)
                                * (180.0 / np.pi)
                            )

                            feat = extract_features_from_window(
                                w_acc,
                                w_gyro,
                                speed_mean=35.0,
                                speed_delta=0.0,
                            )

                            feat["class_label"] = class_idx
                            feat["class_name"] = label.lower()
                            feat["source_dataset"] = "Combined_Anomalies"

                            records_list.append(feat)

                            if len(records_list) >= max_per_class[label.lower()]:
                                break

        except Exception as e:
            print("Error processing", fpath, ":", e)
            continue

    print(
        "Extracted",
        len(bump_records),
        "bump windows and",
        len(pothole_records),
        "pothole windows."
    )

    # Original split logic: shuffle windows and use 70/15/15.
    all_anomaly_records = []

    for records_list in [bump_records, pothole_records]:

        RNG.shuffle(records_list)

        n = len(records_list)
        n_train = int(n * 0.70)
        n_val = int(n * 0.15)

        for i, r in enumerate(records_list):

            if i < n_train:
                r["split"] = "train"
            elif i < n_train + n_val:
                r["split"] = "val"
            else:
                r["split"] = "test"

            all_anomaly_records.append(r)

    return all_anomaly_records


# 8. Normal Road + 8,000-row IMU dataset

The original script adds two types of supplemental data.

### A. Combined Normal

Files in:

```text
dataset/_extracted/combined_normal/
```

are converted to 100-sample windows.

The original script:

- takes up to 200 windows per file;
- uses non-overlapping windows;
- converts gyro rad/s → deg/s;
- assigns class 2 (`normal_driving`);
- uses fixed `speed_mean=45.0` and `speed_delta=0.0`.

### B. Road Accident IMU 8,000

The original script:

- selects up to 500 crash rows;
- converts acceleration from g → m/s²;
- converts gyro rad/s → deg/s;
- tiles each single row across 100 samples;
- adds Gaussian noise;
- supplies synthetic speed delta;
- labels the generated windows as `crash`.

This means these 500 examples are **synthetically expanded from individual rows**, not 500 naturally occurring 1-second sequences.


In [8]:
def process_normal_road_and_imu_csv():
    records = []

    # ========================================================
    # A. Normal Road
    # ========================================================

    print("\nProcessing Isolated Normal Road CSVs...")

    norm_csvs = sorted(
        glob.glob(os.path.join(NORMAL_ROAD_DIR, "*.csv"))
    )

    for fpath in norm_csvs:
        try:
            df = pd.read_csv(fpath)

            acc_cols = [
                "totalAcceleration_x",
                "totalAcceleration_y",
                "totalAcceleration_z",
            ]

            if not all(c in df.columns for c in acc_cols):
                acc_cols = [
                    "accelerometer_x",
                    "accelerometer_y",
                    "accelerometer_z",
                ]

            gyro_cols = [
                "gyroscope_x",
                "gyroscope_y",
                "gyroscope_z",
            ]

            step = WINDOW_SIZE

            # Original limit: up to 200 windows per file.
            for s in range(
                0,
                len(df) - WINDOW_SIZE + 1,
                step
            ):
                if s // step >= 200:
                    break

                w_acc = (
                    df[acc_cols]
                    .iloc[s:s + WINDOW_SIZE]
                    .values
                    .astype(float)
                )

                w_gyro = (
                    df[gyro_cols]
                    .iloc[s:s + WINDOW_SIZE]
                    .values
                    .astype(float)
                    * (180.0 / np.pi)
                )

                feat = extract_features_from_window(
                    w_acc,
                    w_gyro,
                    speed_mean=45.0,
                    speed_delta=0.0
                )

                feat["class_label"] = 2
                feat["class_name"] = "normal_driving"
                feat["source_dataset"] = "Combined_Normal"

                records.append(feat)

        except Exception as e:
            print("Error processing normal", fpath, ":", e)

    # ========================================================
    # B. road_accident_imu_dataset_8000.csv
    # ========================================================

    imu_8000_path = os.path.join(
        DATASET_DIR,
        "road_accident_imu_dataset_8000.csv"
    )

    if os.path.exists(imu_8000_path):

        print(
            "\nProcessing",
            os.path.basename(imu_8000_path),
            "..."
        )

        df8 = pd.read_csv(imu_8000_path)

        crash_sub = df8[df8["Crash_Label"] == 1]

        for i in range(min(500, len(crash_sub))):

            row = crash_sub.iloc[i]

            base_acc = (
                np.array([
                    row["Acc_X"],
                    row["Acc_Y"],
                    row["Acc_Z"],
                ])
                * 9.80665
            )

            # Synthetic 1-second acceleration window.
            w_acc = (
                np.tile(base_acc, (WINDOW_SIZE, 1))
                + RNG.randn(WINDOW_SIZE, 3) * 0.8
            )

            base_gyro = (
                np.array([
                    row["Gyro_X"],
                    row["Gyro_Y"],
                    row["Gyro_Z"],
                ])
                * (180.0 / np.pi)
            )

            # Synthetic 1-second gyro window.
            w_gyro = (
                np.tile(base_gyro, (WINDOW_SIZE, 1))
                + RNG.randn(WINDOW_SIZE, 3) * 2.0
            )

            feat = extract_features_from_window(
                w_acc,
                w_gyro,
                speed_mean=float(row["Speed_kmh"]),
                speed_delta=-float(row["Speed_kmh"]) * 0.8,
            )

            feat["class_label"] = 0
            feat["class_name"] = "crash"
            feat["source_dataset"] = "road_accident_imu_8000"

            records.append(feat)

    # ========================================================
    # Assign train / validation / test
    # ========================================================

    RNG.shuffle(records)

    n = len(records)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    for i, r in enumerate(records):

        if i < n_train:
            r["split"] = "train"
        elif i < n_train + n_val:
            r["split"] = "val"
        else:
            r["split"] = "test"

    return records


# 9. Main pipeline

The original `main()` function executes the following sequence:

### Step 1 — Find VZCrash files

```text
train_data/*.parquet
validation_data/*.parquet
test_data/*.parquet
```

### Step 2 — Process only selected files

The original script uses:

- first 7 VZCrash training files
- first 2 validation files
- first 2 test files

### Step 3 — Add bump/pothole

From `Combined_Anomalies`.

### Step 4 — Add normal road + crash supplement

From `Combined_Normal` and the 8,000-row IMU dataset.

### Step 5 — Combine everything

All generated feature rows are concatenated into one DataFrame.

### Step 6 — Remove rows with missing values

`dropna()`.

### Step 7 — Add sample IDs

```text
SAMP_000001
SAMP_000002
...
```

### Step 8 — Export all artifacts


In [9]:
def main():
    start_time = time.time()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 70)
    print("STARTING MASTER DATASET CONSTRUCTION")
    print("=" * 70)

    # ========================================================
    # 1. VZCrash files
    # ========================================================

    vz_train_files = sorted(
        glob.glob(
            os.path.join(
                DATASET_DIR,
                "VZCrash/train_data/*.parquet"
            )
        )
    )

    vz_val_files = sorted(
        glob.glob(
            os.path.join(
                DATASET_DIR,
                "VZCrash/validation_data/*.parquet"
            )
        )
    )

    vz_test_files = sorted(
        glob.glob(
            os.path.join(
                DATASET_DIR,
                "VZCrash/test_data/*.parquet"
            )
        )
    )

    print("VZCrash train files:", len(vz_train_files))
    print("VZCrash val files  :", len(vz_val_files))
    print("VZCrash test files :", len(vz_test_files))

    # Original selection
    train_records = process_vzcrash_split(
        "train",
        vz_train_files[:7],
        max_per_class=1750
    )

    val_records = process_vzcrash_split(
        "val",
        vz_val_files[:2],
        max_per_class=375
    )

    test_records = process_vzcrash_split(
        "test",
        vz_test_files[:2],
        max_per_class=375
    )

    # ========================================================
    # 2. Road anomalies
    # ========================================================

    anomaly_records = process_road_anomalies(
        max_per_class={
            "bump": 2200,
            "pothole": 2200
        }
    )

    # ========================================================
    # 3. Supplemental normal + 8000 IMU
    # ========================================================

    supplemental_records = process_normal_road_and_imu_csv()

    # ========================================================
    # 4. Combine
    # ========================================================

    all_records = (
        train_records
        + val_records
        + test_records
        + anomaly_records
        + supplemental_records
    )

    master_df = pd.DataFrame(all_records)

    # Defensive check: the original script tried to access
    # master_df['class_name'] even when no records were created.
    # That produces a confusing KeyError. Fail here with the
    # real cause instead.
    if master_df.empty:
        raise RuntimeError(
            "Master dataset contains 0 rows. "
            "No source records were successfully processed. "
            "Check DATASET_DIR, VZCrash file paths, and _extracted folders."
        )

    # Original builder removes rows containing missing values.
    master_df.dropna(inplace=True)

    # Add unique sample ID.
    master_df.insert(
        0,
        "sample_id",
        [f"SAMP_{i+1:06d}" for i in range(len(master_df))]
    )

    print("\n" + "=" * 70)
    print("DATASET COMPILATION COMPLETE")
    print("=" * 70)

    print("Total Windows Extracted:", len(master_df))

    print("\nClass Distribution Across Entire Master Dataset:")
    print(master_df["class_name"].value_counts())

    print("\nSplit Distribution:")
    print(master_df["split"].value_counts())

    print("\nCross-tabulation (Class vs Split):")
    print(
        pd.crosstab(
            master_df["class_name"],
            master_df["split"],
            margins=True
        )
    )

    # ========================================================
    # 5. Save master dataset
    # ========================================================

    print("\nSaving Master Dataset files...")

    master_parquet_path = os.path.join(
        OUTPUT_DIR,
        "master_driving_events.parquet"
    )

    master_csv_path = os.path.join(
        OUTPUT_DIR,
        "master_driving_events.csv"
    )

    master_df.to_parquet(
        master_parquet_path,
        index=False
    )

    master_df.to_csv(
        master_csv_path,
        index=False
    )

    print(
        f"Saved: {master_parquet_path} "
        f"({os.path.getsize(master_parquet_path)/(1024*1024):.2f} MB)"
    )

    print(
        f"Saved: {master_csv_path} "
        f"({os.path.getsize(master_csv_path)/(1024*1024):.2f} MB)"
    )

    # ========================================================
    # 6. Save train / val / test
    # ========================================================

    for sp in ["train", "val", "test"]:

        sp_df = master_df[
            master_df["split"] == sp
        ].copy()

        sp_pqt = os.path.join(
            OUTPUT_DIR,
            f"{sp}.parquet"
        )

        sp_csv = os.path.join(
            OUTPUT_DIR,
            f"{sp}.csv"
        )

        sp_df.to_parquet(
            sp_pqt,
            index=False
        )

        sp_df.to_csv(
            sp_csv,
            index=False
        )

        print(
            f"Saved {sp} split: "
            f"{len(sp_df)} rows -> {sp_pqt}"
        )

    # ========================================================
    # 7. Label metadata
    # ========================================================

    label_meta_path = os.path.join(
        OUTPUT_DIR,
        "label_mapping.json"
    )

    with open(label_meta_path, "w") as f:
        json.dump(
            {
                "label_to_name": LABEL_MAPPING,
                "name_to_label": INV_LABEL_MAPPING,
                "num_classes": len(LABEL_MAPPING),
                "window_size_samples": WINDOW_SIZE,
                "sampling_rate_hz": 100,
                "window_duration_seconds": WINDOW_SIZE * DT,
                "units": {
                    "acceleration": (
                        "m/s^2 (SI, includes gravity like "
                        "Android Sensor.TYPE_ACCELEROMETER)"
                    ),
                    "gyroscope": (
                        "deg/s (converted from Android rad/s "
                        "via 180/pi)"
                    ),
                    "speed": (
                        "km/h (converted from Android "
                        "Location.getSpeed m/s via * 3.6)"
                    ),
                },
            },
            f,
            indent=2
        )

    print("Saved:", label_meta_path)

    # ========================================================
    # 8. Feature metadata
    # ========================================================

    feature_cols = [
        c for c in master_df.columns
        if c not in [
            "sample_id",
            "class_label",
            "class_name",
            "source_dataset",
            "split",
        ]
    ]

    feature_meta_path = os.path.join(
        OUTPUT_DIR,
        "feature_metadata.json"
    )

    with open(feature_meta_path, "w") as f:
        json.dump(
            {
                "num_features": len(feature_cols),
                "feature_names": feature_cols,
            },
            f,
            indent=2
        )

    print(
        f"Saved: {feature_meta_path} "
        f"({len(feature_cols)} features)"
    )

    print(
        f"\nExecution finished successfully in "
        f"{time.time() - start_time:.2f} seconds."
    )

    return master_df


# The original script executes main() only when run as a script.
# In a notebook, keep execution explicit:
#
# master_df = main()


In [10]:
# ============================================================
# DATASET SOURCE INVENTORY — run this before the full builder
# ============================================================

vz_train_files = sorted(
    glob.glob(str(DATASET_DIR / "VZCrash" / "train_data" / "*.parquet"))
)
vz_val_files = sorted(
    glob.glob(str(DATASET_DIR / "VZCrash" / "validation_data" / "*.parquet"))
)
vz_test_files = sorted(
    glob.glob(str(DATASET_DIR / "VZCrash" / "test_data" / "*.parquet"))
)

anomaly_csvs = sorted(
    glob.glob(str(ANOMALIES_DIR / "*.csv"))
    + glob.glob(str(ANOMALIES_DIR / "*" / "*.csv"))
)

normal_csvs = sorted(
    glob.glob(str(NORMAL_ROAD_DIR / "*.csv"))
)

print("=" * 70)
print("DATASET SOURCE INVENTORY")
print("=" * 70)
print("VZCrash train parquet :", len(vz_train_files))
print("VZCrash val parquet   :", len(vz_val_files))
print("VZCrash test parquet  :", len(vz_test_files))
print("Anomaly CSV files     :", len(anomaly_csvs))
print("Normal-road CSV files :", len(normal_csvs))

if vz_train_files:
    print("\nExample VZCrash train file:")
    print(vz_train_files[0])

if anomaly_csvs:
    print("\nExample anomaly file:")
    print(anomaly_csvs[0])

if normal_csvs:
    print("\nExample normal-road file:")
    print(normal_csvs[0])

total_source_files = (
    len(vz_train_files)
    + len(vz_val_files)
    + len(vz_test_files)
    + len(anomaly_csvs)
    + len(normal_csvs)
)

if total_source_files == 0:
    raise RuntimeError(
        "\nNO INPUT FILES WERE FOUND. "
        "Do not run the builder yet. "
        "Check the dataset path printed above and make sure "
        "the VZCrash / _extracted folders contain the expected files."
    )

print("\n✓ At least one source file was found.")


DATASET SOURCE INVENTORY
VZCrash train parquet : 11
VZCrash val parquet   : 3
VZCrash test parquet  : 2
Anomaly CSV files     : 97
Normal-road CSV files : 5

Example VZCrash train file:
/Users/viditagarwal/Desktop/Phone Black Box/dataset/VZCrash/train_data/train-00000-of-00011.parquet

Example anomaly file:
/Users/viditagarwal/Desktop/Phone Black Box/dataset/_extracted/combined_anomalies/1.csv

Example normal-road file:
/Users/viditagarwal/Desktop/Phone Black Box/dataset/_extracted/combined_normal/37 - Normal Road.csv

✓ At least one source file was found.


# 10. Run the complete builder

⚠️ **This cell can take significant time and disk space.**

Run it only after verifying that the dataset directory is correctly configured.

```python
master_df = main()
```

The function returns the final DataFrame so that we can immediately audit it in the notebook.


In [11]:
# Run the complete master dataset builder.
#
# IMPORTANT:
# 1. Run the DATASET SOURCE INVENTORY cell immediately above first.
# 2. If the inventory shows zero source files, fix the dataset path first.
#
master_df = main()

if master_df.empty:
    raise RuntimeError(
        "The builder produced 0 rows. "
        "Check the source inventory and processing messages above."
    )

print("\nFinal shape:", master_df.shape)


STARTING MASTER DATASET CONSTRUCTION
VZCrash train files: 11
VZCrash val files  : 3
VZCrash test files : 2

Processing VZCrash train (7 files, target ~1750/class)...
  Reading file 1/7: train-00000-of-00011.parquet
  Reading file 2/7: train-00001-of-00011.parquet
  Reading file 3/7: train-00002-of-00011.parquet
  Reading file 4/7: train-00003-of-00011.parquet
  Reading file 5/7: train-00004-of-00011.parquet
  Reading file 6/7: train-00005-of-00011.parquet
  Reading file 7/7: train-00006-of-00011.parquet
VZCrash train summary: {0: 1750, 1: 1750, 2: 1193, 3: 1750, 4: 1750, 5: 1750, 6: 1750}

Processing VZCrash val (2 files, target ~375/class)...
  Reading file 1/2: validation-00000-of-00003.parquet
  Reading file 2/2: validation-00001-of-00003.parquet
VZCrash val summary: {0: 375, 1: 375, 2: 246, 3: 375, 4: 375, 5: 375, 6: 375}

Processing VZCrash test (2 files, target ~375/class)...
  Reading file 1/2: test-00000-of-00002.parquet
  Reading file 2/2: test-00001-of-00002.parquet
VZCrash t

# 11. Validate the generated master dataset

After building the dataset, we should not immediately start model training.

First verify:

- feature count
- class distribution
- source distribution
- missing values
- infinite values
- duplicate IDs
- split sizes
- class × split distribution

These checks tell us whether the builder actually produced the dataset we intended.


In [12]:
def audit_master_dataset(master_df):
    metadata_cols = [
        "sample_id",
        "class_label",
        "class_name",
        "source_dataset",
        "split",
    ]

    feature_cols = [
        c for c in master_df.columns
        if c not in metadata_cols
    ]

    print("=" * 70)
    print("MASTER DATASET AUDIT")
    print("=" * 70)

    print("Shape:", master_df.shape)
    print("Feature count:", len(feature_cols))

    print("\nClass distribution:")
    print(master_df["class_name"].value_counts())

    print("\nSource distribution:")
    print(master_df["source_dataset"].value_counts())

    print("\nSplit distribution:")
    print(master_df["split"].value_counts())

    print("\nClass × split:")
    print(pd.crosstab(
        master_df["class_name"],
        master_df["split"]
    ))

    print("\nDuplicate sample IDs:",
          master_df["sample_id"].duplicated().sum())

    print("Missing feature values:",
          master_df[feature_cols].isna().sum().sum())

    print("Infinite feature values:",
          np.isinf(master_df[feature_cols].to_numpy()).sum())

    print("\nFeature names:")
    for i, name in enumerate(feature_cols, 1):
        print(f"{i:02d}. {name}")

    print("\n✓ Audit complete.")

# Example:
# audit_master_dataset(master_df)


# 12. What the script does well

The original builder has several good engineering decisions:

### 1. Fixed-size windows

Every sample becomes a consistent 1-second representation.

### 2. Physical unit normalization

Different datasets are brought into common sensor units before feature extraction.

### 3. Deterministic randomness

`RandomState(42)` makes random selection reproducible.

### 4. Unified ontology

Different datasets are mapped into the same nine classes.

### 5. Metadata artifacts

The builder exports:

- `label_mapping.json`
- `feature_metadata.json`

These are important for reproducing model input and mobile deployment.

### 6. Multiple feature families

The representation combines:

- time-domain statistics
- acceleration magnitude
- jerk
- angular velocity
- speed
- FFT energy
- spectral entropy
- skewness
- kurtosis
- cross-axis correlations


# 13. Important limitations of the original builder

This section is intentionally included because the notebook should explain **what the code actually did**, not hide potential weaknesses.

## A. The 51-feature comment is incorrect

The implementation creates 62 features.

The metadata generated by the script correctly calculates:

```python
len(feature_cols)
```

so the final `feature_metadata.json` should report 62.

## B. Some source splits are not recording-group safe

For `Combined_Anomalies`, overlapping windows are generated from the same annotated block and then shuffled into train/validation/test.

That can allow highly similar windows from the same physical event to appear in different splits.

For a production-quality ML evaluation, a future builder should split by **recording/event first**, then create windows.

## C. Some speed values are synthetic/default

For isolated anomalies:

```python
speed_mean=35.0
speed_delta=0.0
```

For normal road:

```python
speed_mean=45.0
speed_delta=0.0
```

These values are assumptions made by the original builder, not measured speed from those recordings.

## D. The 8,000-row crash data is synthetically expanded

Each selected crash row becomes a 100-sample sequence by tiling one sensor reading and adding Gaussian noise.

Therefore these examples should be clearly identified as **synthetically expanded crash windows**.

## E. VZCrash maneuver labels are partly derived using thresholds

The original source class 2 is inspected and some windows are re-labelled as:

- hard braking
- braking
- sharp turn
- turn
- normal driving

using threshold rules.

Those rules should be treated as **heuristic pseudo-labeling**, not independent ground truth.

## F. Feature semantics must remain identical during mobile inference

The Phone Black Box application must reproduce:

- the same 100-sample window
- the same sensor units
- the same feature calculations
- the same feature order

Otherwise the trained model will receive a different feature distribution than it saw during training.


# 14. Recommended next version of the dataset builder

Do **not** change the current master dataset blindly if your objective is reproducibility.

Instead, create a documented **v2 builder** with these improvements:

```text
Raw recordings
      ↓
Recording-level split
      ↓
Window extraction
      ↓
Feature extraction
      ↓
Class balancing
      ↓
Quality checks
      ↓
Master dataset
```

The most important improvement is:

> **Never split overlapping windows randomly across train/validation/test.**

The recording/event should be the grouping unit.

That will give a more trustworthy estimate of how well Phone Black Box generalizes to a previously unseen recording.


# 15. Final artifact contract

The builder produces:

```text
dataset/
└── master_dataset/
    ├── master_driving_events.parquet
    ├── master_driving_events.csv
    │
    ├── train.parquet
    ├── train.csv
    ├── val.parquet
    ├── val.csv
    ├── test.parquet
    ├── test.csv
    │
    ├── label_mapping.json
    └── feature_metadata.json
```

The ML notebook should consume these files rather than re-processing raw sensor data.

That creates a clean project architecture:

```text
01_master_dataset_builder.ipynb
            ↓
02_dataset_audit.ipynb
            ↓
03_driving_event_classifier.ipynb
            ↓
04_model_export_and_mobile_inference.ipynb
```

This separation makes your B.Tech project easier to explain, reproduce, debug and defend during evaluation.
